In [4]:
# STEP 1: COMMENTS
# Project  : Restaurant Tips Analysis

# STEP 2: IMPORTS & FILE I/O

import csv
import os
from functools import reduce

# create folders
os.makedirs("data/raw",       exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

# download dataset
import urllib.request
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv"
urllib.request.urlretrieve(url, "data/raw/tips.csv")
print("Dataset downloaded!")

# read csv file
data = []
with open("data/raw/tips.csv", "r") as f:
    reader = csv.DictReader(f)
    for row in reader:
        data.append(row)

print(f"Total rows loaded : {len(data)}")
print(f"First row         : {data[0]}")

# STEP 3: TYPE CONVERSION + CLEANING

def clean_record(row):
    # type conversion
    total_bill = float(row["total_bill"])
    tip        = float(row["tip"])
    size       = int(row["size"])

    # conditional statements — skip bad rows
    if total_bill <= 0: return None
    if tip < 0:         return None
    if size < 1:        return None

    # calculate tip percentage
    tip_pct = round((tip / total_bill) * 100, 2)

    # string operations — .strip() removes spaces
    return {
        "total_bill" : total_bill,
        "tip"        : tip,
        "sex"        : row["sex"].strip(),
        "smoker"     : row["smoker"].strip(),
        "day"        : row["day"].strip(),
        "time"       : row["time"].strip(),
        "size"       : size,
        "tip_pct"    : tip_pct
    }

# clean all rows
cleaned_data = []
for row in data:
    result = clean_record(row)
    if result is not None:
        cleaned_data.append(result)

print(f"\nRaw rows   : {len(data)}")
print(f"Clean rows : {len(cleaned_data)}")
print(f"Dropped    : {len(data) - len(cleaned_data)}")

# STEP 4: LISTS + TUPLES + DICTIONARIES + LOOPS

# group by day using dictionary + loop
day_groups = {}

for record in cleaned_data:
    day = record["day"]
    if day not in day_groups:
        day_groups[day] = []
    day_groups[day].append(record)

# calculate avg tip per day
day_order = ["Thur", "Fri", "Sat", "Sun"]   # list
day_stats = []                               # list of tuples

for day in day_order:
    records  = day_groups[day]
    tip_list = []

    for r in records:
        tip_list.append(r["tip_pct"])

    avg_tip = round(sum(tip_list) / len(tip_list), 2)

    # store as tuple
    result = (day, len(records), avg_tip)
    day_stats.append(result)

# print table
print("\n--- Insight 1: Day-wise Tip Analysis ---")
print(f"{'Day':<6} {'Visits':>8} {'Avg Tip%':>10}")
print("-" * 28)

for stat in day_stats:
    day, count, avg = stat        # unpack tuple
    print(f"{day:<6} {count:>8} {avg:>9}%")

# find best day
best = day_stats[0]
for stat in day_stats:
    if stat[2] > best[2]:
        best = stat

print(f"\n✅ Best tipping day : {best[0]}")
print(f"   Avg Tip %       : {best[2]}%")
print(f"   Total Visits    : {best[1]}")

# STEP 5: STRINGS + FUNCTIONS + LAMBDA

# helper functions
def get_values(records, key):
    values = []
    for r in records:
        values.append(r[key])
    return values

def calc_mean(numbers):
    return round(sum(numbers) / len(numbers), 2)

def print_title(title):
    print(f"\n{'='*40}")
    print(f"  {title.upper()}")        # string .upper()
    print(f"{'='*40}")

# lambda functions
is_dinner = lambda r: r["time"] == "Dinner"
is_small  = lambda r: r["size"] <= 2
get_tip   = lambda r: r["tip_pct"]

# overall stats
all_tips  = get_values(cleaned_data, "tip_pct")
all_bills = get_values(cleaned_data, "total_bill")

print_title("overall stats")
print(f"  Total Records : {len(cleaned_data)}")
print(f"  Avg Tip %     : {calc_mean(all_tips)}%")
print(f"  Avg Bill      : ${calc_mean(all_bills)}")

# party size analysis
size_groups = {}
for record in cleaned_data:
    s = record["size"]
    if s not in size_groups:
        size_groups[s] = []
    size_groups[s].append(record)

print_title("insight 2: party size analysis")
print(f"{'Size':<8} {'Visits':>8} {'Avg Tip%':>10} {'Avg Bill':>10}")
print("-" * 40)

for size in sorted(size_groups.keys()):
    records  = size_groups[size]
    avg_tip  = calc_mean(get_values(records, "tip_pct"))
    avg_bill = calc_mean(get_values(records, "total_bill"))
    print(f"{size:<8} {len(records):>8} {avg_tip:>9}% ${avg_bill:>8.2f}")

# lambda to split small vs large
small_list = [r for r in cleaned_data if is_small(r)]
large_list = [r for r in cleaned_data if not is_small(r)]

small_avg  = calc_mean(get_values(small_list, "tip_pct"))
large_avg  = calc_mean(get_values(large_list, "tip_pct"))

print(f"\n  Small party avg tip : {small_avg}%")
print(f"  Large party avg tip : {large_avg}%")
print(f"\n✅ Small parties tip {round(small_avg - large_avg, 2)}% more!")

# STEP 6: MAP + FILTER + REDUCE

# filter → separate dinner and lunch
dinner = list(filter(lambda r: r["time"] == "Dinner", cleaned_data))
lunch  = list(filter(lambda r: r["time"] == "Lunch",  cleaned_data))

# map → extract bills
dinner_bills = list(map(lambda r: r["total_bill"], dinner))
lunch_bills  = list(map(lambda r: r["total_bill"], lunch))

# reduce → total revenue
dinner_rev = reduce(lambda a, b: a + b, dinner_bills)
lunch_rev  = reduce(lambda a, b: a + b, lunch_bills)

# total tips using reduce
total_tips = reduce(lambda a, r: a + r["tip"], cleaned_data, 0)

print_title("insight 3: lunch vs dinner")
print(f"  Dinner visits  : {len(dinner)}")
print(f"  Lunch  visits  : {len(lunch)}")
print(f"  Dinner revenue : ${round(dinner_rev, 2)}")
print(f"  Lunch  revenue : ${round(lunch_rev,  2)}")
print(f"  Total  tips    : ${round(total_tips, 2)}")
print(f"\n✅ Dinner is {round(dinner_rev / lunch_rev, 1)}x more revenue than Lunch!")

# STEP 7: OOP — TipAnalyser CLASS

class TipAnalyser:

    def __init__(self, records):       # constructor
        self.records = records
        self.total   = len(records)

    def avg_tip(self):                 # method 1
        tips = [r["tip_pct"] for r in self.records]
        return round(sum(tips) / len(tips), 2)

    def best_day(self):                # method 2
        days = {}
        for r in self.records:
            d = r["day"]
            if d not in days:
                days[d] = []
            days[d].append(r["tip_pct"])
        best = max(days, key=lambda d: sum(days[d]) / len(days[d]))
        return best

    def total_revenue(self):           # method 3
        return round(sum(r["total_bill"] for r in self.records), 2)

    def __str__(self):                 # print summary
        return (
            f"\n{'='*35}\n"
            f"  Total Records : {self.total}\n"
            f"  Avg Tip %     : {self.avg_tip()}%\n"
            f"  Best Day      : {self.best_day()}\n"
            f"  Total Revenue : ${self.total_revenue()}\n"
            f"{'='*35}"
        )


# create object
analyser = TipAnalyser(cleaned_data)
print(analyser)

# STEP 8: SAVE PROCESSED FILE (File I/O)

fields = list(cleaned_data[0].keys())

with open("data/processed/tips_cleaned.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    writer.writerows(cleaned_data)

print("\n💾 Cleaned data saved to data/processed/tips_cleaned.csv")

# STEP 9: VISUAL SUMMARY (ASCII Dashboard)

def ascii_bar(value, max_val, width=25):
    filled = int((value / max_val) * width)
    return "█" * filled + "░" * (width - filled)

print_title("visual summary dashboard")

# day wise chart
print("\n  📊 Day-wise Avg Tip %")
print("  " + "-" * 38)
max_tip = max(stat[2] for stat in day_stats)
for stat in day_stats:
    bar = ascii_bar(stat[2], max_tip)
    print(f"  {stat[0]:<5} | {bar} {stat[2]}%")

# party size chart
print("\n  📊 Party Size Avg Tip %")
print("  " + "-" * 38)
size_avgs = []
for size in sorted(size_groups.keys()):
    avg = calc_mean(get_values(size_groups[size], "tip_pct"))
    size_avgs.append((size, avg))

max_s = max(s[1] for s in size_avgs)
for size, avg in size_avgs:
    bar = ascii_bar(avg, max_s)
    print(f"  Sz {size} | {bar} {avg}%")

# lunch vs dinner chart
print("\n  📊 Revenue: Lunch vs Dinner")
print("  " + "-" * 38)
max_r = max(dinner_rev, lunch_rev)
print(f"  Din  | {ascii_bar(dinner_rev, max_r)} ${round(dinner_rev)}")
print(f"  Lun  | {ascii_bar(lunch_rev,  max_r)} ${round(lunch_rev)}")




Dataset downloaded!
Total rows loaded : 244
First row         : {'total_bill': '16.99', 'tip': '1.01', 'sex': 'Female', 'smoker': 'No', 'day': 'Sun', 'time': 'Dinner', 'size': '2'}

Raw rows   : 244
Clean rows : 244
Dropped    : 0

--- Insight 1: Day-wise Tip Analysis ---
Day      Visits   Avg Tip%
----------------------------
Thur         62     16.13%
Fri          19     16.99%
Sat          87     15.31%
Sun          76     16.69%

✅ Best tipping day : Fri
   Avg Tip %       : 16.99%
   Total Visits    : 19

  OVERALL STATS
  Total Records : 244
  Avg Tip %     : 16.08%
  Avg Bill      : $19.79

  INSIGHT 2: PARTY SIZE ANALYSIS
Size       Visits   Avg Tip%   Avg Bill
----------------------------------------
1               4     21.73% $    7.24
2             156     16.57% $   16.45
3              38     15.21% $   23.28
4              37     14.59% $   28.61
5               5     14.15% $   30.07
6               4     15.62% $   34.83

  Small party avg tip : 16.7%
  Large party av